In [0]:
%load_ext autoreload
%autoreload 2
# Enables autoreload; learn more at https://docs.databricks.com/en/files/workspace-modules.html#autoreload-for-python-modules
# To disable autoreload; run %autoreload 0

In [0]:
from pyspark.sql import SparkSession

from src.ingestion.detect_delimiter import detect_delimiter
from src.ingestion.detect_header import detect_header
from src.ingestion.parse_file import parse_file
from src.transformation.create_dataframe import create_dataframe
from src.processing.clean_text import clean_text
from src.processing.extract_skills import extract_skills
from src.matching.find_overlap import find_overlap
from pyspark.sql.functions import col, udf, array, explode
from pyspark.sql.types import ArrayType
from pyspark.sql.types import StructType, StructField, StringType, IntegerType

spark = SparkSession.builder.appName("pipeline_runner").getOrCreate()

profile_raw = [
    "Generated by HR System",
    "Export Date: 2026-06-20",
    "Department: Data Analytics",
    "",
    "profile_id;name;experience;skills;;;;",
    "1;John Doe;Built ETL pipelines with Spark and Airflow;python,sql,spark;;;;",
    "2;Alice Smith;Worked on SQL dashboards and Power BI;sql,excel,powerbi;;;;",
    "3;Bob Lee;Developed ML models using Python and Pandas;python,pandas,machine learning;;;;",
    "4;Sara Khan;Built data lake pipelines in Databricks;spark,databricks,airflow;;;;"
]

job_raw = [
    "Job Export System",
    "Generated: 2026-06-20",
    "",
    "job_id;title;skills;;;;",
    "101;Data Engineer;python,sql,spark,airflow;;;;",
    "102;BI Analyst;sql,excel,tableau;;;;",
    "103;ML Engineer;python,machine learning,pandas;;;;",
    "104;Platform Engineer;spark,databricks,deltalake;;;;"
]



# -----------------------------
# 1. PROFILE PIPELINE
# -----------------------------
delimiter_profile = detect_delimiter(profile_raw, [',', ';', '\t', '|'])
delimiter_profile_val = delimiter_profile["result"]["delimiter"]

header_profile = detect_header(profile_raw, delimiter_profile_val)

parsed_profile = parse_file(profile_raw, delimiter_profile, header_profile)
profile_records = parsed_profile["result"]["data"]

for pr in profile_records:
    if "profile_id" in pr:
        pr["profile_id"] = int(pr["profile_id"])
    else:
        print("Missing profile_id row:", pr)

schema = StructType([
    StructField("profile_id", IntegerType(), True),
    StructField("name", StringType(), True),
    StructField("experience", StringType(), True)
])

profile_df = create_dataframe(profile_records, spark, schema)["result"]["dataframe"]

cleaned_profile = clean_text(profile_df, "experience")["result"]["dataframe"]
profile_skills_df = extract_skills(cleaned_profile, "experience")["result"]["dataframe"]


# -----------------------------
# 2. JOB PIPELINE
# -----------------------------
delimiter_job = detect_delimiter(job_raw, [',', ';', '\t', '|'])
delimiter_job_val = delimiter_job["result"]["delimiter"]

header_job = detect_header(job_raw, delimiter_job_val)

parsed_job = parse_file(job_raw, delimiter_job, header_job)
job_records = parsed_job["result"]["data"]

for jr in job_records:
    if "job_id" in jr:
        jr["job_id"] = int(jr["job_id"])
    else:
        print("Missing job_id row:", jr)

job_schema = StructType([
    StructField("job_id", IntegerType(), True),
    StructField("title", StringType(), True),
    StructField("skills", StringType(), True)
])

job_df = create_dataframe(job_records, spark, schema=job_schema)["result"]["dataframe"]

cleaned_job = clean_text(job_df, "skills")["result"]["dataframe"]
job_skills_df = extract_skills(cleaned_job, "skills")["result"]["dataframe"]


# -----------------------------
# 3. MATCHING LAYER
# -----------------------------

overlap_result = find_overlap(profile_skills_df, job_skills_df)

display(overlap_result["result_df"])
